# Lab: bounded local security exercises
Scope: synthetic users, strings, an in-memory database, and safe path/HTML checks. Never send these markers anywhere.


In [ ]:
import html, os, sqlite3
print('Python environment ready; local-only scope active')


## Objectives
Map assets and trust boundaries; reproduce ownership, SQL, mass-assignment, HTML, path, and secret mistakes; then test root-cause defenses.


## Baseline reproduction — Predict 1
If `get_doc` checks only that a document ID exists, can Ben read Ana's synthetic document? Predict before running.


In [ ]:
docs = {1: {'id':1, 'owner':'ana', 'title':'Private plan', 'body':'synthetic'}}
def vulnerable_get(doc_id, actor):
    return {'status': 200, 'doc': docs[doc_id]} if doc_id in docs else {'status':404}
assert vulnerable_get(1, 'ben')['status'] == 200
print('Baseline reproduces broken access control locally')


**Pre-edit hypothesis:** checking ownership before returning the document will deny Ben while preserving Ana's access. Asset: private document; actors: Ana/Ben; boundary: API input to document store.


## Predict 2
If a SQL value is supplied through a `?` parameter, will a quote marker become SQL syntax? No; SQLite receives it as data.


In [ ]:
db = sqlite3.connect(':memory:')
db.execute('CREATE TABLE docs (id INTEGER, owner TEXT, body TEXT)')
db.executemany('INSERT INTO docs VALUES (?,?,?)', [(1,'ana','A'),(2,'ben','B')])
marker = "ana' OR owner='ben"
rows = db.execute('SELECT id, owner FROM docs WHERE owner = ?', (marker,)).fetchall()
assert rows == []
print('Parameterized value stayed data')


## Predict 3
If a caller submits `owner` in an update dictionary, should a document update accept it? No. Mass assignment must allowlist writable fields.


In [ ]:
def secure_get(doc_id, actor):
    doc = docs.get(doc_id)
    if doc is None or doc['owner'] != actor: return {'status':404}
    return {'status':200, 'doc':doc}
WRITABLE = {'title','body'}
def update(doc, fields):
    for key in WRITABLE:
        if key in fields: doc[key] = fields[key]
    return doc
assert secure_get(1, 'ben')['status'] == 404
assert secure_get(1, 'ana')['status'] == 200
update(docs[1], {'owner':'ben','title':'New'})
assert docs[1]['owner'] == 'ana' and docs[1]['title'] == 'New'


## Guided TODO: safe output and paths
Try to choose a safe output function and a path rule before reading the assertions. We do not render HTML or read files. The next code cell is the executable reference solution.


In [ ]:
def safe_html(text):
    return html.escape(text, quote=True)
def safe_relative(root, candidate):
    root_abs = os.path.abspath(root)
    target = os.path.abspath(os.path.join(root_abs, candidate))
    return target if os.path.commonpath([root_abs, target]) == root_abs else None
untrusted = '<b>synthetic</b>'
assert safe_html(untrusted) == '&lt;b&gt;synthetic&lt;/b&gt;'
assert safe_relative('/tmp/training-root', 'notes/a.txt').startswith('/tmp/training-root')
assert safe_relative('/tmp/training-root', '../outside.txt') is None


HTML escaping changes markup characters into text. The path check compares a canonical absolute path with an allowlisted root. Neither is a complete production policy: context, symlinks, platform behavior, and approved destinations still need review.


## Intentionally weak AI-style fixes
Filtering one known SQL marker or one `..` string is not a root-cause defense. Parameterization and canonical root checks address the interpreter and path boundaries.


## Independent challenge — attempt before checking
Add a variant check for a denied owner, a quote marker, an HTML ampersand, and a path with `../`. Write the expected safe outcomes first, then compare with the executable checks below.


In [ ]:
assert secure_get(1, 'ben')['status'] == 404
assert db.execute('SELECT id FROM docs WHERE owner = ?', ("ben' OR 1=1",)).fetchall() == []
assert safe_html('&') == '&amp;'
assert safe_relative('/tmp/training-root', '../x') is None
print('Variant defenses passed')


## Exit questions
1. Who enforces authorization?
2. What does parameterization prove?
3. What remains unproved?

### Answers
1. The server at the document boundary.
2. Values are not parsed as SQL syntax for this query.
3. Full browser CSRF policy, SSRF network policy, symlink races, dependency vulnerabilities, and production permissions.

## Evidence handoff
Save the scope statement, threat map, baseline/after outputs, secret review, AI diff critique, and residual-risk owner decisions.
